# BiRefNet → ONNX 変換ノートブック

BiRefNet の公式事前学習重み（`.pth`）をダウンロードし、ONNX へエクスポートします。

## 実行環境
- Google Colab （ランタイム GPU 不要、CPU でも OK）
- **`imsize=1344` のような高解像度はメモリを多く使うため、ハイメモリランタイム推奨**
- `imsize=1024` なら通常ランタイムでも可

## 出力
- `birefnet_<imsize>x<imsize>.onnx`

---
参考：
- BiRefNet: https://github.com/ZhengPeng7/BiRefNet
- deform_conv2d_onnx_exporter: https://github.com/masamitsu-murase/deform_conv2d_onnx_exporter

## 1. パラメータ

変換したい入力解像度をここで 1 箇所だけ設定します。

In [ ]:
IMSIZE = 1024  # 1024 または 1344 など

WEIGHT_URL = "https://github.com/ZhengPeng7/BiRefNet/releases/download/v1/BiRefNet-general-epoch_244.pth"
WEIGHT_FILENAME = "BiRefNet-general-epoch_244.pth"
ONNX_FILENAME = f"birefnet_{IMSIZE}x{IMSIZE}.onnx"

## 2. セットアップ

BiRefNet 本体と、ONNX エクスポーター（DeformConv2d オペレータを ONNX に出すために必要）を取得し、事前学習重みをダウンロードします。

In [ ]:
%cd /content/
!git clone https://github.com/ZhengPeng7/BiRefNet.git
%cd /content/BiRefNet

# DeformConv2d を ONNX にエクスポートするためのヘルパー
!git clone https://github.com/masamitsu-murase/deform_conv2d_onnx_exporter
!cp deform_conv2d_onnx_exporter/src/deform_conv2d_onnx_exporter.py .

# 事前学習重み
!wget -nc --content-disposition {WEIGHT_URL}

## 3. 依存パッケージ

BiRefNet の `requirements.txt` と、ONNX 関連をインストールします。

**`numpy==1.23.5` にダウングレードさせた後、セッションのランタイムを再起動してから次に進みます。**
（Colab のメニュー “ランタイム → セッションを再起動”）

In [ ]:
!pip uninstall -q torchaudio torchdata torchtext -y
!pip install -q -r requirements.txt
!pip install -q onnx onnxruntime

# ONNX エクスポート時のシェイプ推論互換性ケア
!pip install -q numpy==1.23.5

## 4. deform_conv2d_onnx_exporter のパッチ

動的 shape を含むテンソルで `_get_tensor_dim_size()` が `None` を返しエクスポートが落ちる事象への暫定対応。
ストライド情報から表を復元して返すように書き換えます。

In [ ]:
PATCH_TARGET = "return sym_help._get_tensor_dim_size(tensor, dim)"
PATCH_REPLACEMENT = '''
    tensor_dim_size = sym_help._get_tensor_dim_size(tensor, dim)
    if tensor_dim_size is None and (dim == 2 or dim == 3):
        import typing
        from torch import _C
        x_type = typing.cast(_C.TensorType, tensor.type())
        x_strides = x_type.strides()
        tensor_dim_size = x_strides[2] if dim == 3 else x_strides[1] // x_strides[2]
    elif tensor_dim_size is None and dim == 0:
        import typing
        from torch import _C
        x_type = typing.cast(_C.TensorType, tensor.type())
        x_strides = x_type.strides()
        tensor_dim_size = x_strides[3]
    return tensor_dim_size
'''

with open('deform_conv2d_onnx_exporter.py', 'r') as fp:
    src = fp.read()

if PATCH_TARGET in src:
    src = src.replace(PATCH_TARGET, PATCH_REPLACEMENT)
    with open('deform_conv2d_onnx_exporter.py', 'w') as fp:
        fp.write(src)
    print('patched')
else:
    print('already patched (skip)')

## 5. ONNX へ変換

PyTorch 重みをロードして `torch.onnx.export` で出力します。

In [ ]:
%cd /content/BiRefNet
import torch
from utils import check_state_dict
from models.birefnet import BiRefNet
import deform_conv2d_onnx_exporter

model = BiRefNet(bb_pretrained=False)
state_dict = check_state_dict(torch.load(WEIGHT_FILENAME, map_location='cpu'))
model.load_state_dict(state_dict)
model.to('cpu').eval()

torch.set_float32_matmul_precision('high')

# DeformConv2d を ONNX に出すためのオペレータ登録
deform_conv2d_onnx_exporter.register_deform_conv2d_onnx_op()

dummy_input = torch.randn(1, 3, IMSIZE, IMSIZE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_FILENAME,
    opset_version=16,
    input_names=['input_image'],
    output_names=['output_image'],
    verbose=False,
)

print(f'saved: {ONNX_FILENAME}')

## 6. （任意）動作確認

生成された ONNX で推論が通るかをチェックします。サンプル画像は任意の JPEG/PNG を `sample.jpg` としてアップロードしてください。

In [ ]:
import copy
import cv2
import numpy as np
import onnxruntime

SAMPLE_PATH = 'sample.jpg'  # 任意の画像パス

image = cv2.imread(SAMPLE_PATH)
assert image is not None, f'画像が見つかりません: {SAMPLE_PATH}'
image_height, image_width = image.shape[:2]

# 前処理
x = cv2.resize(image, (IMSIZE, IMSIZE))
x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
x = (x / 255.0 - mean) / std
x = x.transpose(2, 0, 1).astype('float32')[None]

# 推論
session = onnxruntime.InferenceSession(
    ONNX_FILENAME,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
outputs = session.run(None, {session.get_inputs()[0].name: x})

# 後処理（マスク生成）
def sigmoid(v):
    return 1.0 / (1.0 + np.exp(-v))

mask = sigmoid(np.squeeze(outputs[-1])) * 255
mask = mask.astype('uint8')
mask = cv2.resize(mask, (image_width, image_height))

# 切り抜き画像
extracted = np.where(np.stack([mask] * 3, axis=-1) > 127, image, 255)

# Colab なら表示
try:
    from google.colab.patches import cv2_imshow
    cv2_imshow(mask)
    cv2_imshow(extracted)
except ImportError:
    cv2.imwrite('mask.png', mask)
    cv2.imwrite('extracted.png', extracted)
    print('saved: mask.png, extracted.png')